In [1]:
import pandas as pd
import psycopg2

In [3]:
dbname = 'dw_eletronicos'
user = 'postgres'
password = 'XXXXXX'
host = 'localhost'
port = '5432'

conexao = psycopg2.connect(dbname=dbname, user=user, password=password, host=host, port=port)
cursor = conexao.cursor()

cursor.execute("""

                   CREATE TABLE IF NOT EXISTS dw.fato_vendas
                        (   
                            pedido_id int,
                            data_pedido date,
                            cliente_id int,
                            produto_id int,
                            id_canal_venda int,
                            id_pagamento int,
                            id_localidade int,
                            quantidade int,
                            preco_unitario decimal(10,2),
                            desconto decimal(10,2),
                            frete decimal(10,2),
                            valor_total decimal(10,2),
                            status_pedido varchar(50),
               
                            PRIMARY KEY (pedido_id, produto_id),

                            foreign key (pedido_id) references dw.dim_pedido(pedido_id),
                            foreign key (data_pedido) references dw.dim_tempo(data),
                            foreign key (cliente_id) references dw.dim_cliente (cliente_id),
                            foreign key (produto_id) references dw.dim_produto (produto_id),
                            foreign key (id_canal_venda) references dw.dim_canal_venda (id_canal_venda),
                            foreign key (id_pagamento) references dw.dim_pagamento (id_pagamento),
                            foreign key (id_localidade) references dw.dim_localidade (id_localidade)
                        );
               
                   


               """)
conexao.commit()
cursor.close()
conexao.close()

In [6]:
dbname = 'dw_eletronicos'
user = 'postgres'
password = 'XXXXXX'
host = 'localhost'
port = '5432'

conexao = psycopg2.connect(dbname=dbname, user=user, password=password, host=host, port=port)
cursor = conexao.cursor()

cursor.execute("""

            WITH base AS (
                            SELECT
                                v.pedido_id,
                                dt.data AS data_pedido,
                                v.cliente_id,
                                p.produto_id,
                                dc.id_canal_venda,
                                dp.id_pagamento,
                                l.id_localidade,
                                v.quantidade,
                                v.preco_unitario,
                                v.desconto,
                                v.frete,
                                v.valor_total,
                                v.status_pedido,

                                ROW_NUMBER() OVER (
                                    PARTITION BY v.pedido_id, v.produto_id
                                    ORDER BY v.data_pedido DESC
                                ) AS rn

                            FROM staging.stg_vendas v

                            JOIN dw.dim_tempo dt ON v.data_pedido = dt.data
                            JOIN dw.dim_produto p ON v.produto_id = p.produto_id
                            JOIN dw.dim_pagamento dp ON TRIM(UPPER(v.forma_pagamento)) = TRIM(UPPER(dp.forma_pagamento))
                            JOIN dw.dim_canal_venda dc ON TRIM(UPPER(v.canal_vendas)) = TRIM(UPPER(dc.canal_venda))
                            JOIN dw.dim_localidade l ON TRIM(UPPER(v.cidade)) = TRIM(UPPER(l.cidade))
                                                    AND TRIM(UPPER(v.estado)) = TRIM(UPPER(l.estado))
                        )

                        INSERT INTO dw.fato_vendas (
                            pedido_id,
                            data_pedido,
                            cliente_id,
                            produto_id,
                            id_canal_venda,
                            id_pagamento,
                            id_localidade,
                            quantidade,
                            preco_unitario,
                            desconto,
                            frete,
                            valor_total,
                            status_pedido
                        )
                        SELECT
                            pedido_id,
                            data_pedido,
                            cliente_id,
                            produto_id,
                            id_canal_venda,
                            id_pagamento,
                            id_localidade,
                            quantidade,
                            preco_unitario,
                            desconto,
                            frete,
                            valor_total,
                            status_pedido
                        FROM base
                        WHERE rn = 1
                        ON CONFLICT (pedido_id, produto_id)
                        DO UPDATE SET
                            quantidade = EXCLUDED.quantidade,
                            valor_total = EXCLUDED.valor_total,
                            status_pedido = EXCLUDED.status_pedido;
                           
               """)
               


               

conexao.commit()
cursor.close()
conexao.close()

In [4]:
dbname = 'dw_eletronicos'
user = 'postgres'
password = 'XXXXXX'
host = 'localhost'
port = '5432'

conexao = psycopg2.connect(dbname=dbname, user=user, password=password, host=host, port=port)
cursor = conexao.cursor()

cursor.execute("""


                        CREATE TABLE IF NOT EXISTS dw.fato_estoque
                            (
                                produto_id integer primary key,
                                estoque_inicial int,
                                entradas_periodo int,
                                estoque_atual int,
                                estoque_minimo int,
                                custo_unitario decimal(10,2),
                                foreign key (produto_id) references dw.dim_produto(produto_id)
                                
                            );
                """)
conexao.commit()
cursor.close()
conexao.close()

In [7]:
dbname = 'dw_eletronicos'
user = 'postgres'
password = 'XXXXXX'
host = 'localhost'
port = '5432'

conexao = psycopg2.connect(dbname=dbname, user=user, password=password, host=host, port=port)
cursor = conexao.cursor()


cursor.execute("""      
               
               INSERT INTO dw.fato_estoque 
                    (
                        produto_id,
                        estoque_inicial,
                        entradas_periodo,
                        estoque_atual,
                        estoque_minimo,
                        custo_unitario
                    )
                    SELECT
                        dp.produto_id,
                        e.estoque_inicial,
                        e.entradas_periodo,
                        e.estoque_atual,
                        e.estoque_minimo,
                        e.custo_unitario
                    FROM staging.stg_estoque e 
                    INNER JOIN dw.dim_produto dp 
                        ON e.produto_id = dp.produto_id
                    WHERE NOT EXISTS (
                        SELECT 1
                        FROM dw.fato_estoque f
                        WHERE f.produto_id = dp.produto_id
                    );

               """)
               

conexao.commit()
cursor.close()
conexao.close()

In [5]:
dbname = 'dw_eletronicos'
user = 'postgres'
password = 'XXXXXX'
host = 'localhost'
port = '5432'

conexao = psycopg2.connect(dbname=dbname, user=user, password=password, host=host, port=port)
cursor = conexao.cursor()

cursor.execute("""                


                        CREATE TABLE IF NOT EXISTS dw.fato_devolucoes
                            (
                                devolucao_id serial primary key,
                                pedido_id int,
                                cliente_id int,
                                produto_id int,
                                data_devolucao date,
                                quantidade_devolvida int,
                                valor_devolvido decimal(10,2),
                                motivo_devolucao varchar(100),
                                status_devolucao varchar(30),
                                foreign key (pedido_id) references dw.dim_pedido(pedido_id),
                                foreign key (cliente_id) references dw.dim_cliente (cliente_id),
                                foreign key (produto_id) references dw.dim_produto (produto_id),
                                foreign key (data_devolucao) references dw.dim_tempo (data)

                                );
                """)
conexao.commit()
cursor.close()
conexao.close()

In [8]:
dbname = 'dw_eletronicos'
user = 'postgres'
password = 'XXXXXX'
host = 'localhost'
port = '5432'

conexao = psycopg2.connect(dbname=dbname, user=user, password=password, host=host, port=port)
cursor = conexao.cursor()


cursor.execute("""                 
                     INSERT INTO dw.fato_devolucoes (
                            pedido_id,
                            cliente_id,
                            produto_id,
                            data_devolucao,
                            quantidade_devolvida,
                            valor_devolvido,
                            motivo_devolucao,
                            status_devolucao
                        )
                        SELECT
                            v.pedido_id,
                            c.cliente_id,
                            p.produto_id,
                            d.data_devolucao,
                            d.quantidade_devolvida,
                            d.valor_devolvido,
                            d.motivo_devolucao,
                            d.status_devolucao
                        FROM staging.stg_devolucoes d

                        JOIN dw.fato_vendas v
                            ON d.pedido_id = v.pedido_id

                        JOIN dw.dim_cliente c
                            ON d.cliente_id = c.cliente_id

                        JOIN dw.dim_produto p
                            ON d.produto_id = p.produto_id

                        WHERE NOT EXISTS (
                            SELECT 1
                            FROM dw.fato_devolucoes f
                            WHERE f.pedido_id = d.pedido_id
                            AND f.produto_id = d.produto_id
                            AND f.data_devolucao = d.data_devolucao
                        );
                """)

conexao.commit()
cursor.close()
conexao.close()